# Iranian Stock Market — Benchmark-Relative Portfolio Optimization

**Research question:** Can a long-only, fully invested allocation maximize *historical mean active return* while limiting each training-period shortfall to `k` relative to the benchmark?

This reproducible notebook uses the **same Pyomo/GLPK linear program** as the original coursework, but removes four duplicated workflows and distinguishes **in-sample fit** from a **chronological held-out evaluation**. It is a research exercise, **not investment advice**.

## 1. Experiment configuration

Source format: original Excel workbooks `Dataset1.xlsx` ... `Dataset4.xlsx`, with **instruments in rows, chronological prices in columns**, and the **benchmark in the final row**. The Excel workbooks are **not provided**. Place your own workbooks in `data/raw/`, or use the clearly marked *synthetic* demo.

`k=0.02` means a two-percentage-point **per-period** return shortfall limit *only on the fitting data*. It does **not** guarantee a future downside bound.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Works from repo root or notebooks directory, including VS Code and Jupyter.
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'scripts' / 'run_portfolio.py').exists() else CWD.parent
assert (ROOT / 'scripts' / 'run_portfolio.py').exists(), 'Open this notebook inside the repository.'

DATA_FILES = sorted((ROOT / 'data' / 'raw').glob('Dataset*.xlsx'))
if not DATA_FILES:
    DATA_FILES = [ROOT / 'data' / 'example' / 'synthetic_price_matrix.csv']
    print('DEMO MODE: No original dataset files found. Using SYNTHETIC sample; not Iranian market data.')
else:
    print(f'Loaded {len(DATA_FILES)} original-format workbook paths.')

K = 0.02
TRAIN_FRACTION = 0.70
OUTPUT_DIR = ROOT / 'results'
print('Selected files:', [f.name for f in DATA_FILES])

## 2. Mathematical formulation

Let $r_{it}$ be the asset return, $b_t$ the benchmark return, and $w_i$ the portfolio weight:

$$\max_w\quad \frac{1}{T}\sum_{t=1}^T\left(\sum_i w_i r_{it}-b_t\right)$$

subject to

$$\sum_i w_i=1,\qquad w_i\geq0,\qquad \sum_i w_i r_{it}\geq b_t-k\quad\forall t\in\text{fit periods}.$$

This is a linear program because the historical benchmark is fixed. `GLPK` solves it via `Pyomo`. The original study fitted and displayed performance on the same 116 return periods. The updated workflow separately reports a chronological train/test split where test observations do **not** determine portfolio weights.

## 3. Run all selected workbooks

Three reports are produced per workbook: `in_sample_full`, `chronological_train`, and `chronological_test`. The latter reuses the *train-fit weights* without reoptimization. Inputs are neither downloaded nor fabricated in real-data mode.

In [ ]:
import subprocess
cmd = [sys.executable, str(ROOT / 'scripts' / 'run_portfolio.py'),
       '--files', *[str(p) for p in DATA_FILES],
       '--k', str(K), '--train-fraction', str(TRAIN_FRACTION),
       '--output-dir', str(OUTPUT_DIR)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 4. Review summary and plots

Do not compare *in-sample* results with out-of-sample results as if both measured generalization. `active_floor_breach_count` may be **positive** on held-out periods; the floor is enforced only on fitting periods.

In [ ]:
from IPython.display import display, Image
summary = pd.read_csv(OUTPUT_DIR / 'summary.csv')
display(summary.round(5))
for f in DATA_FILES:
    plot = OUTPUT_DIR / f.stem / 'chronological_test_wealth.png'
    print(f.name, 'chronological held-out comparison')
    display(Image(filename=str(plot)))

## 5. Interpretation and limitations

- **Dataset status:** The included workbook is synthetic only; historical Dataset1–4 Excel files were not provided.
- **Fit vs. evaluation:** The original four optimal LP outcomes were in-sample; no out-of-sample performance has been claimed from them. See `docs/HISTORICAL_RESULTS.md`.
- **Market conventions:** Corporate actions, price adjustment, missing trading sessions, benchmark composition, the meaning of price units, and true observation frequency are not verified.
- **Model assumptions:** Static long-only weights, no borrowing, no turnover or transaction costs, and a two-percentage-point historical active-return floor.
- **Research extension:** Consider rolling-forward backtests, regularization/turnover costs, scenario stress tests, and sensitivity analysis over `k` before making any financial claim.